# 장기 계획(long-term plan) API — 출력 미리보기

`POST /v1/plan/generate` (submit) + `GET /v1/plan/generate/{job_id}` (poll) 가
어떤 모양으로 나오는지 확인하는 노트북.

내부적으로는 기존 todo **planner** 파이프라인(planner LoRA)을 그대로 재사용한다.
기존 `/v1/todo/chat` 과 다른 점은 응답에 **일자별 `plan`** 과 `goal_tag` 가 노출된다는 것뿐이다.

여기서는 RunPod 없이 **scripted FakeLLM** 으로 파이프라인을 돌려 응답 스키마를 보여준다.
프로젝트 루트(`mongle-ai/`)에서 `uv run jupyter` 로 열어 실행한다.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # mongle-ai/ 를 import 경로에 추가

from dataclasses import dataclass
from datetime import date, datetime, timedelta

from agents.todo_creation.planner.pipeline import PlannerPorts, run
from agents.todo_creation.schemas import PlannerInput, TaskCandidate

TODAY = date(2026, 6, 19)
DEADLINE = date(2026, 9, 19)  # 3개월 뒤

## Scripted FakeLLM

`LLMPort` 를 구현해 "3개월 안에 토익 800점" 목표에 대한 6일치 plan 을 흉내 낸다.
실제로는 planner LoRA 가 이 자리를 채운다 (`adapters/todo_creation/runpod_llm.py`).

In [ ]:
@dataclass
class FakeLLM:
    """planner LoRA 대역. judge -> plan 으로 한 번에 충분 판정."""

    async def judge_sufficiency(self, *, history, message, today, user_profile_memory=None):
        goal = {'intent': 'plan', 'goal_text': '토익 800점', 'goal_tag': '토익', 'deadline': DEADLINE}
        return True, [], goal

    async def generate_follow_up_question(self, *, missing_aspects, history):
        return '언제까지 목표를 이루고 싶으세요?'

    async def generate_plan(self, *, parsed_goal, today):
        steps = [
            '기출 1세트 풀기', 'LC 파트1~2', 'RC 파트5 단어', 'LC 파트3~4',
            'RC 파트6 독해', '약점 파트 복습',
        ]
        plan = [
            {'date': today + timedelta(days=i),
             'tasks': [TaskCandidate(title=title, due_date=today + timedelta(days=i))]}
            for i, title in enumerate(steps)
        ]
        summary = '3개월 안에 토익 800점을 위해 매일 LC/RC 를 번갈아 학습하는 계획이야.'
        return summary, plan

    async def generate_goal_tag(self, *, parsed_goal, history):
        return parsed_goal.get('goal_tag', '목표')

    async def tag_plan(self, *, plan, parsed_goal):
        return plan

    async def split_tasks(self, *, prompt, today):
        ...

ports = PlannerPorts(llm=FakeLLM())

In [ ]:
req = PlannerInput(user_id='u1', message='3개월 안에 토익 800점', today=TODAY)
result = await run(req, ports=ports, now=datetime(2026, 6, 19, 9, 0))

# poll 응답 본문(GET /v1/plan/generate/{job_id})의 result 가 이 모양이다.
print(result.model_dump_json(indent=2, exclude_none=True))

## 사람이 읽기 좋게 렌더링

`plan` = 일자별 분해, `todos` = 오늘(due == today), `calendar_events` = 미래 날짜로 자동 분리된다.

In [ ]:
print(f'goal_tag: {result.goal_tag}')
print(f'summary : {result.summary_text}')
print(f'thread  : {result.thread_id}')
print('-' * 40)
for day in result.plan or []:
    titles = ', '.join(t.title for t in day.tasks)
    print(f'{day.date}  {titles}')
print('-' * 40)
print(f'오늘 todos        : {[t.title for t in result.todos]}')
print(f'calendar_events  : {[(str(e.due_date), e.title) for e in result.calendar_events]}')

## 실제 HTTP 호출 (참고)

```bash
# 1) submit -> 202, result.job_id 반환
curl -s -XPOST localhost:8000/v1/plan/generate \
  -H 'X-API-Key: $MONGLE_AI_API_KEY' -H 'Content-Type: application/json' \
  -d '{"user_id":"u1","goal":"3개월 안에 토익 800점","today":"2026-06-19"}'

# 2) poll -> status: pending | done | error. done 이면 result 가 위 스키마.
curl -s localhost:8000/v1/plan/generate/<job_id> -H 'X-API-Key: $MONGLE_AI_API_KEY'
```

정보가 부족하면(예: 마감일 없음) `result.kind == "follow_up"` 으로 되묻고,
응답의 `thread_id` 를 다음 submit 에 넣어 대화를 이어간다.